In [1]:
import h5py
import numpy as np

def load_and_prepare(path, pilot_syms, p_spacing):
    # 1) load & squeeze
    with h5py.File(path, "r") as f:
        x = np.squeeze(f["x"][...])   # → (N, 14, 612)
        y = np.squeeze(f["y"][...])   # → (N, 14, 612)
        h = np.squeeze(f["h"][...])   # → (N, 14, 612)

    # sanity
    assert x.shape == y.shape == h.shape
    N, Ns, Nsc = x.shape

    # 2) LS estimate at pilots
    H_ls = y / x
    pilot_subcs = np.arange(0, Nsc, p_spacing)  # e.g. [0,4,8,…]

    # pick out just the pilots: (N, len(pilot_syms), len(pilot_subcs))
    H_ls_p = H_ls[:, pilot_syms, :][:, :, pilot_subcs]

    # 3) interpolate back to every subcarrier
    all_subcs = np.arange(Nsc)
    H_interp = np.zeros_like(H_ls)  # complex64

    for n in range(N):
        for si, sym in enumerate(pilot_syms):
            # real part
            real_vals = H_ls_p[n, si, :].real
            H_interp[n, sym, :].real = np.interp(all_subcs, pilot_subcs, real_vals)
            # imag part
            imag_vals = H_ls_p[n, si, :].imag
            H_interp[n, sym, :].imag = np.interp(all_subcs, pilot_subcs, imag_vals)

        # (optional) to fill non-pilot OFDM-symbol rows as well,
        # you'd do a 2D interpolation over (symbol, subcarrier) here.

    # 4) stack into real/imag channels
    inputs = np.stack([H_interp.real, H_interp.imag], axis=-1)  # (N,14,612,2)
    labels = np.stack([h.real,        h.imag],        axis=-1)  # (N,14,612,2)

    return inputs, labels


In [6]:
pilot_syms = [3, 10]
p_spacing  = 4
inputs, labels = load_and_prepare("data_snr15.hdf5", pilot_syms, p_spacing)

In [7]:
print(f"inputs mean:  {inputs.mean():.6e}")
print(f"inputs std:   {inputs.std():.6e}")
print(f"labels mean:  {labels.mean():.6e}")
print(f"labels std:   {labels.std():.6e}")

inputs mean:  7.610566e-05
inputs std:   2.675870e-01
labels mean:  5.231963e-04
labels std:   7.071070e-01


In [5]:
print(f"inputs mean:  {inputs.mean():.6e}")
print(f"inputs std:   {inputs.std():.6e}")
print(f"labels mean:  {labels.mean():.6e}")
print(f"labels std:   {labels.std():.6e}")

inputs mean:  4.616916e-06
inputs std:   2.656181e-01
labels mean:  1.056763e-05
labels std:   7.071057e-01


In [3]:
err2 = np.abs(inputs - labels)**2

# overall MSE scalar
mse = np.mean(err2)

print(f"Baseline LS-interpolation MSE: {mse:.6e}")

Baseline LS-interpolation MSE: 4.293675e-01
